In [2]:
import pandas as pd
import re
import plotly.express as px
import plotly.subplots as sp


# Identificación de los tickers presentes en las noticias

Si bien los habíamos tomado los Stock_symbol asociados a cada noticia como punto de partida en la creación del subdataset, no quisimos utilizarlos como labels, sino que preferimos implementar un proceso propio para la detección de los tickers, pensando en poder generar a futuro una metodología de trabajo replicable a mayor escala.

Sin embargo, para simplificar el proceso, decidimos utilizar los mencionados tickers como "filtro inicial". Nuestra premisa es que en un proyecto a mayor esacala, este "filtro inicial" vendría dado por un listado de tickers de interés sobre los cuales se desee realizar el análisis. 

Nuestro objetivo fue buscar detectar cuáles de las noticias de nuestro dataset estaban relacionadas con estos "tickers de interes" y, para ello, nos enfocamos en buscar aquellos que aparecían explicitamente en las mismas (aún en los casos donde una noticia mencionara más de un ticker).

Si bien intentamos varias metodologías alternativas, finalmente nos decidimos por la siguiente:

1) Con el listado de "tickers de interés", buscamos cuáles de ellos aparecen en el texto de las noticias utilizando una función Regex. Nuestro foco fue buscar “Tickers aislados", es decir, que no tuviera uno o más caracteres alfanumérico inmediatamente antes y que, a su vez, estuvieran seguidos por una "separación" (ya sea un espacio, un signo de puntuación). 
Es decir, nos enfocamos en buscar que el ticker no sea un substring dentro de otra palabra para evitar falsos positivos.
2) Al tener certeza de que el dataset es de noticias financieras, consideramos que no eran necesarias desambiguaciones.
3) Si bien hicimos pruebas enriqueciendo la búsqueda con los datos adicionales de Company Name, no se observaron mejoras en los resultados al usarlos como un string completo y, al particionarlo por palabras, obtubimos muchos falsos positivos.
5) Revisamos manualmente una muestra pequeñade los resultados para verificar que la metodología era adecuada. Creemos que sería bueno mejorar esta técnica de validación manual en el futuro con alguna metodología semi-automática, sin embargo, en pos de avanzar con el proyecto decidimos dejarlo así por ahora.

In [3]:
#Listado inicial de tickes pre-seleccionados para el proyecto = "tickers de interes"

tickers = [
    "AAPL", "AMD", "AMZN", "BRK", "BSMRCGRO", "DIS", "DNOVEAOA", "F", "FDEV",
    "FFEB", "FSMB", "GDMA", "GOOG", "GS", "GSEE", "HCRB", "KJUL", "KO", "MSFT",
    "NVDA", "PFFL", "PMAY", "SPY", "TSLA", "UBER", "UCIB", "UFEB", "UOCT",
    "WINC", "WLDR", "WMT", "WTRE"
]

## Preparacion del dataframe de noticias para el proceso de búsqueda de tickers

In [4]:
df_news = pd.read_csv("nasdaq_subdataset_proyecto.csv")
df_news.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125261 entries, 0 to 125260
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Date           125261 non-null  object
 1   Article_title  125261 non-null  object
 2   Stock_symbol   125261 non-null  object
 3   Url            125261 non-null  object
 4   Article        125261 non-null  object
dtypes: object(5)
memory usage: 4.8+ MB


Como consideramos que el ticker puede encontrarse en tanto en el campo Article como Article_title o Url, vamos a concatenar estos tres campos en un nuevo campo llamado texto_full para efectuar la búsqueda.

In [6]:
df_news["texto_full"] = df_news["Article_title"] + " " + df_news["Url"] + " " + df_news["Article"]

## Detección de tickers en las noticias (campo texto_full) utilizando la función regex

In [7]:
#Armado de los patrones regex
puntuacion = r"\s.,;:!?\-_()\"'"
patterns = {
    ticker: re.compile(rf"(?<!\w){ticker}(?=[{puntuacion}]|\b)")
    for ticker in tickers
}


#Construcción de la función de búsqueda de tickers para aplicar al DataFrame
def encontrar_tickers(texto):
    if not isinstance(texto, str):
        return []
    encontrados = [ticker for ticker, pat in patterns.items() if pat.search(texto)]
    return encontrados

#Aplicamos la función al campo texto_full del DataFrame 
df_news["tickers_encontrados"] = df_news["texto_full"].apply(encontrar_tickers)

#Filtramos las filas con al menos un ticker encontrado para construir un nuevo subdataset de news (df_con_tickers)
df_con_tickers = df_news[df_news["tickers_encontrados"].map(len) > 0]

# Printeamos algunos ejemplo
print(df_con_tickers[["texto_full", "tickers_encontrados"]].head())

                                          texto_full tickers_encontrados
0  UK antitrust regulator wins appeal over Apple ...              [AAPL]
1  Japan aircon king Daikin looks to custom chips...        [AAPL, AMZN]
2  Judge set to rule on Berkshire Hathaway reques...              [AAPL]
3  Analysts predict more brands will flee X after...   [AAPL, DIS, TSLA]
4  Netflix (NFLX) to Offer Grand Theft Auto Trilo...        [AAPL, MSFT]


In [96]:
df_con_tickers.head(2)

,Date,Article_title,Stock_symbol,Url,Article,texto_full,tickers_encontrados
0,2023-11-30 00:00:00+00:00,UK antitrust regulator wins appeal over Apple ...,AAPL,https://www.nasdaq.com/articles/uk-antitrust-r...,Adds details from ruling and CMA comment in pa...,UK antitrust regulator wins appeal over Apple ...,[AAPL]
1,2023-11-30 00:00:00+00:00,Japan aircon king Daikin looks to custom chips...,AAPL,https://www.nasdaq.com/articles/japan-aircon-k...,"By Sam Nussey and Miho Uranaka\nTOKYO, Dec 1 (...",Japan aircon king Daikin looks to custom chips...,"[AAPL, AMZN]"


In [ ]:
#Como mencionamos al inicio del notebook, revisamos manualmente varios casos al azar 
#para ver si los tickers fueron correctamente detectados y vimos resultados positivos.

#Dejamos un ejemplo a continuación a modo de referencia:

print(df_con_tickers.iloc[3].loc['texto_full'])

#[AAPL, DIS, TSLA]


Analysts predict more brands will flee X after Musk tirade https://www.nasdaq.com/articles/analysts-predict-more-brands-will-flee-x-after-musk-tirade-0 By Chavi Mehta and Jaspreet Singh
Nov 30 (Reuters) - More advertisers are likely to flee Elon Musk's social-media company X after the billionaire lashed out at some of the biggest names in the media industry at a New York Times DealBook event for dropping out of the platform, analysts said on Thursday.
Walt Disney DIS.N and Warner Bros. Discovery WBD.Osuspended advertising on X earlier this month following Musk's endorsement of an antisemitic post that falsely claimed members of the Jewish community were stoking hatred against white people.
After apologizing for his post at the event on Wednesday, Musk unleashed a profanity-laced tirade against some of the advertisers for fleeing the platform.
The Tesla TSLA.O chief also acknowledged that an extended boycott by advertisers could bankrupt X, formerly Twitter, but suggested that the publi

In [8]:
df_con_tickers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42295 entries, 0 to 125256
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Date                 42295 non-null  object
 1   Article_title        42295 non-null  object
 2   Stock_symbol         42295 non-null  object
 3   Url                  42295 non-null  object
 4   Article              42295 non-null  object
 5   texto_full           42295 non-null  object
 6   tickers_encontrados  42295 non-null  object
dtypes: object(7)
memory usage: 2.6+ MB


## Graficos sobre el nuevo subdataset para observar la distribución de noticias por ticker y por fecha

### Total de noticias por ticker

In [9]:
#Tenemos que hacer un "explode del DF" para poder graficar correctamente la distribución de noticias por ticker (ya que, como vimos, en muchos casos, una noticia 
#está asociada a más de un ticker)
df_to_expand = df_con_tickers.copy()

df_expanded = df_to_expand.explode("tickers_encontrados")

# Contamos apariciones por ticker
conteo_single = (
    df_expanded["tickers_encontrados"]
    .value_counts()
    .reset_index()
)
conteo_single.columns = ["Ticker", "Cantidad"]

# Grafico
fig= px.bar(
    conteo_single,
    x="Cantidad",
    y="Ticker",
    orientation="h",
    title="Cantidad de noticias por ticker detectado",
    template='plotly_white',
    text='Cantidad'
)

# Estilo visual
fig.update_traces(
    marker_color="#eb990c"

)

# Ajustes para legibilidad
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=400 + 20 * len(df_expanded["tickers_encontrados"].unique()),
    xaxis_title='Cantidad de noticias',
    yaxis_title='Símbolo',
    font=dict(size=12)
)

fig.show()


Decidimos eliminar los tickers UCIB, PMAY y WTRE por el bajo volumen de noticias para las cuales se detectaron los tickers.

In [11]:
tickers_a_eliminar = {"UCIB", "PMAY", "WTRE"}

#Función para "limpiar" cada fila
def limpiar_tickers(row):
    tickers = list(row["tickers_encontrados"])
    tickers_filtrados = [t for t in tickers if t not in tickers_a_eliminar]
    return tickers_filtrados if tickers_filtrados else None

#Aplicar la "limpieza"
df_con_tickers_filtrado = df_con_tickers.copy()
df_con_tickers_filtrado["tickers_encontrados"] = df_con_tickers_filtrado.apply(limpiar_tickers, axis=1)

#Eliminar las filas que quedaron sin tickers (es decir, aquellas que solo contenían uno o más de los tickers a eliminar)
df_con_tickers_filtrado = df_con_tickers_filtrado.dropna(subset=["tickers_encontrados"]).reset_index(drop=True)



### Cantidad de tickers por noticia

In [12]:
df_multi = df_con_tickers_filtrado.copy()

# Contamos cuántas noticias tienen 1, 2, 3, 4... tickers
conteo_multi = (
    df_multi["tickers_encontrados"]
    .map(len)
    .value_counts()
    .sort_index()
    .reset_index()
)
conteo_multi.columns = ["Cantidad de tickers", "Noticias"]

# Gráfico
fig = px.bar(
    conteo_multi,
    x="Noticias",
    y="Cantidad de tickers",
    orientation="h",
    title="Noticias con múltiples tickers",
    template='plotly_white',
    text="Noticias"
)

# Estilo visual
fig.update_traces(
    marker_color="#eb990c"

)

# Ajustes para legibilidad
fig.update_layout(
    yaxis={'categoryorder': 'total ascending', 'autorange':'reversed'},
    height=400 + 20 * len(df_expanded["tickers_encontrados"].unique()),
    xaxis_title='Cantidad de noticias',
    yaxis_title='Símbolo',
    font=dict(size=12)
)

fig.update_layout(yaxis_title="Cantidad de tickers en la noticia")

fig.show()


### Cantidad de noticias por día por ticker

In [13]:
# Le datos formato a la columna de fecha
df_con_tickers_filtrado['Date'] = pd.to_datetime(df_con_tickers_filtrado['Date'], errors='coerce')

# Agrupamos por día y por símbolo
df_counts = (
    df_con_tickers_filtrado
    .groupby([df_con_tickers_filtrado['Date'].dt.to_period('D'), 'Stock_symbol'])
    .size()
    .reset_index(name='count')
)

# Convertimos el período a datetime para que Plotly lo interprete bien
df_counts['Date'] = df_counts['Date'].dt.to_timestamp()

# Ordenamos por símbolo y fecha para consistencia
df_counts = df_counts.sort_values(['Stock_symbol', 'Date'])

# Lista de símbolos únicos
symbols = df_counts['Stock_symbol'].unique()

# Generamos Subplots: uno por símbolo
fig = sp.make_subplots(
    rows=len(symbols),
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{s}" for s in symbols]
)

# Generamos un gráfico de barras por símbolo
for i, sym in enumerate(symbols, start=1):
    df_sym = df_counts[df_counts['Stock_symbol'] == sym]

    fig.add_bar(
        x=df_sym['Date'],
        y=df_sym['count'],
        name=sym,
        marker_color="#eb990c",
        row=i,
        col=1
    )

    # Eje X: ticks mensuales y etiquetas visibles
    fig.update_xaxes(
        type="date",
        showticklabels=True,
        ticks="outside",
        tickformat="%d-%m-%Y",
        #dtick="M1",
        tickangle=45,
        row=i,
        col=1
    )

# Layout final
fig.update_layout(
    height=250 * len(symbols),
    showlegend=False,
    template="plotly_white",
    title="Cantidad de noticias por día (por símbolo)"
)

fig.show()

C:\Users\Ceci\AppData\Local\Temp\ipykernel_10424\137130063.py:7: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



In [14]:
df_con_tickers_filtrado.to_csv('subds_con_tickers_para_fase2_v2.csv', index=False)

In [15]:
df_con_tickers_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42267 entries, 0 to 42266
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   Date                 42267 non-null  datetime64[ns, UTC]
 1   Article_title        42267 non-null  object             
 2   Stock_symbol         42267 non-null  object             
 3   Url                  42267 non-null  object             
 4   Article              42267 non-null  object             
 5   texto_full           42267 non-null  object             
 6   tickers_encontrados  42267 non-null  object             
dtypes: datetime64[ns, UTC](1), object(6)
memory usage: 2.3+ MB
